# R19-H208 - Cross-unit conversion for the value comparator

**Round** R19 | **Hypothesis** H208 | **Graph** neo4j2 (READ-ONLY) | **Compute** CPU-only, deterministic, no LLM

Extends the H194 deterministic value comparator with dimension-aware unit conversion (mass, length,
pressure, flow; static table, ~2% rounding tolerance). Three clauses:

- **(a)** the H204 unit-variant "conflicts" (3.5 lbs vs 1.6 kg; inches vs mm) reclassify as consistent
- **(b)** ZERO verdict changes on the frozen H194 (60 pairs) and H196 (48 pairs) benches - replay both
- **(c)** does conversion credit >= 1 additional gold in probes-wide-v2-h195.json whose value the graph stores in a converted unit? (reported either way)

**Refuted only if** conversion introduces ANY false credit on the frozen benches.
Harness is the pinned H207 canonical spec (render_fingerprint `96ab16d299fbbc71`).

## Setup - CPU-only, neo4j2 pinned (explicit driver, NOT the .env default instance)

In [1]:
# CPU-only; pin retrieval to neo4j2 via an explicit driver (DEF-5: never inherit the .env default)
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""                     # CPU-only
import re, json, pickle, hashlib, itertools, unicodedata, datetime
from pathlib import Path
from collections import Counter, defaultdict
from neo4j import GraphDatabase
from dotenv import dotenv_values
from knowledge_graph_foundry import load_settings
from knowledge_graph_foundry.graph.graphrag import vector_query
from rich import print as rprint

ROOT = Path("..")
NEO4J2 = "bolt://user-konrad.jelen-kgf-neo4j2:7687"         # 172.19.0.9, pinned baseline, READ-ONLY
_env = dotenv_values(ROOT / ".env")
PW = os.environ.get("NEO4J_PASSWORD") or _env.get("NEO4J_PASSWORD", "kgfoundry")
AUTH = ("neo4j", PW)
settings = load_settings(ROOT / "config.yml")
VEC = settings.graphrag.vector_index_name
K64, RETRIEVE_TOPK, REL_LIMIT = 64, 128, 15
driver2 = GraphDatabase.driver(NEO4J2, auth=AUTH)           # every vector_query uses THIS driver
rprint(f"[cyan]config[/cyan] vec={VEC} eval_k={K64} retrieve_top_k={RETRIEVE_TOPK} rel_cap={REL_LIMIT} (CPU-only, neo4j2 pinned, READ-ONLY)")

2026-07-07 21:25:56.131 | INFO     | knowledge_graph_foundry.config:<module>:40 - PROJ_ROOT path is: /home/lab/workspace/learning/projects/knowledge-graph-foundry


config vec=kgf_entity_embeddings eval_k=64 retrieve_top_k=128 rel_cap=15 (CPU-only, neo4j2 pinned, READ-ONLY)

## Graph pull from neo4j2 + production-faithful render primitives (H207 verbatim)

In [2]:
# Mirrors pipeline._retrieve_local entity_blocks: name/aka/description/Properties/Relations.
with driver2.session() as s:
    ents = s.run("MATCH (e:Entity) RETURN e.id AS id, e.name AS name, e.description AS description, "
                 "properties(e) AS props, labels(e) AS types").data()
    edges = s.run("MATCH (a:Entity)-[r]-(b:Entity) WHERE type(r)<>'SIMILAR_TO' AND a.id<b.id "
                  "RETURN DISTINCT a.id AS a, b.id AS b, type(r) AS rel").data()
    prop_rows = s.run("MATCH (p:Proposition)-[:ABOUT]->(e:Entity) RETURN e.id AS eid, p.text AS text").data()
    alias_rows = s.run("MATCH (e:Entity)-[:SAME_AS*1..2]-(a:Entity) WHERE e.id<>a.id "
                       "RETURN e.id AS eid, collect(DISTINCT a.id)[..5] AS aliases").data()
    emb_head = {r["id"]: r["head"] for r in s.run(
        "MATCH (e:Entity) WHERE e.embedding IS NOT NULL RETURN e.id AS id, e.embedding[0..8] AS head").data()}
node = {r["id"]: r for r in ents}; names = {r["id"]: r["name"] for r in ents}
props_by = defaultdict(list); [props_by[r["eid"]].append(r["text"]) for r in prop_rows]
alias_by = {r["eid"]: r["aliases"] for r in alias_rows}
rels_by = defaultdict(list)
for e in edges:
    rels_by[e["a"]].append((e["rel"], e["b"])); rels_by[e["b"]].append((e["rel"], e["a"]))

def spec_of(r): return {k.removeprefix("prop_"): v for k, v in r["props"].items() if k.startswith("prop_")}
def merged_spec(nid):
    r = node[nid]; spec = dict(spec_of(r))
    for a in [a for a in alias_by.get(nid, []) if a in node]:
        for k, v in spec_of(node[a]).items(): spec.setdefault(k, v)
    return spec
def base_render(nid):
    r = node[nid]; spec = merged_spec(nid); al = [a for a in alias_by.get(nid, []) if a in node]
    aka = (f"Also known as: {', '.join(names.get(a, '') for a in al)}\n" if al else "")
    return f"## {r['name']} ({', '.join(r['types'])})\n{aka}{r['description'] or ''}\nProperties: {json.dumps(spec, default=str)}"
def seed_render(nid):
    rels = "; ".join(f"{t} -> {names.get(b, '')}" for t, b in rels_by.get(nid, [])[:REL_LIMIT])
    return base_render(nid) + "\nRelations: " + rels
def units_of(ids): return [seed_render(n) for n in ids] + [t for n in ids for t in props_by.get(n, [])]
rprint(f"[green]pulled neo4j2[/green] entities {len(node)} edges {len(edges)} propositions {len(prop_rows)}")

pulled neo4j2 entities 2798 edges 3905 propositions 19654

## Scorers - H194 router comparator (verbatim) + deterministic presence

In [3]:
# ---- scorers: H194 router (verbatim) + deterministic presence (exact U word-overlap) ----
_TM = dict.fromkeys(map(ord, "®™©"), None)
def gnorm(s):
    s = (s or "").translate(_TM); s = unicodedata.normalize("NFKC", s)
    s = s.replace(" ", " ").replace("×", "x").replace("*", "x").replace("·", "x")
    s = re.sub(r"(?<=\d),(?=\d)", "", s)
    return re.sub(r"\s+", " ", s.casefold()).strip()
UNITWORD = {"mm":"len_mm","cm":"len_cm","g":"mass_g","kg":"mass_kg","oz":"mass_oz","ml":"vol_ml","l":"vol_l",
   "db":"sound_db","dba":"sound_db","w":"power_w","hz":"freq_hz","cmh2o":"press","m":"alt_m",
   "min":"time_min","mins":"time_min","minute":"time_min","minutes":"time_min","year":"warr_y","years":"warr_y"}
FAM_EQ = {"len_mm":{"len_mm"},"len_cm":{"len_cm"},"alt_m":{"alt_m"},"mass_g":{"mass_g"},"mass_kg":{"mass_kg"},
   "mass_oz":{"mass_oz"},"vol_ml":{"vol_ml","vol_l"},"sound_db":{"sound_db"},"power_w":{"power_w"},
   "time_min":{"time_min"},"warr_y":{"warr_y"},"press":{"press"},"freq_hz":{"freq_hz"}}
def key_family(k):
    k = k.lower()
    if "dimension" in k or re.search(r"_mm\b", k) or "length_mm" in k: return "len_mm"
    if "altitude" in k: return "alt_m"
    if k.endswith("_kg") or "weight_kg" in k: return "mass_kg"
    if re.search(r"_g\b", k): return "mass_g"
    if "_oz" in k: return "mass_oz"
    if re.search(r"_ml\b", k) or "capacity_ml" in k or "water" in k: return "vol_ml"
    if "sound" in k or re.search(r"_db\b", k) or "noise" in k: return "sound_db"
    if "power" in k or "consumption" in k: return "power_w"
    if "ramp" in k or "delay" in k: return "time_min"
    if "warranty" in k: return "warr_y"
    if "pressure" in k: return "press"
    return None
def nums_in(v): return re.findall(r"\d+(?:\.\d+)?", str(v).replace(",", ""))
def ctx_quantities(ids):
    Q = set()
    for nid in ids:
        spec = merged_spec(nid); unit_for = {}
        for k, v in spec.items():
            if k.endswith("_unit"):
                fam = UNITWORD.get(gnorm(str(v)).replace(" ", ""))
                if fam: unit_for[k[:-5]] = fam
        for k, v in spec.items():
            fam = key_family(k) or unit_for.get(k)
            if fam:
                for n in nums_in(v): Q.add((n, fam))
        text = gnorm(seed_render(nid) + " " + " ".join(props_by.get(nid, [])))
        for m in re.finditer(r"(\d+(?:\.\d+)?)\s*(mm|cm|dba|db\(a\)|db|kg|oz|ml|cmh2o|hz|mins|minutes|minute|min|years|year|w|g|l|m)\b", text):
            fam = UNITWORD.get(m.group(2).replace("(a)", ""))
            if fam: Q.add((m.group(1), fam))
    return Q
def parse_gold(gold):
    g = gnorm(gold)
    if re.search(r"\d+\s*x\s*\d+\s*x\s*\d+", g): return ("dim", re.findall(r"\d+(?:\.\d+)?", g))
    if "sd card" in g: return ("sdcard", None)
    m = re.search(r"(\d+(?:\.\d+)?)\s*(mm|cm|dba|db\(a\)|db|kg|oz|ml|cmh2o|cm h2o|hz|mins|minutes|minute|min|years|year|w|g|l|m)\b", g)
    rng = re.search(r"(\d+(?:\.\d+)?)\s*(?:to|-)\s*(\d+(?:\.\d+)?)", g)
    if m:
        u = m.group(2).replace("(a)", "").replace("cm h2o", "cmh2o").replace(" ", ""); fam = UNITWORD.get(u)
        if rng and rng.group(2): return ("range", (rng.group(1), rng.group(2), fam))
        return ("num", (m.group(1), fam))
    if rng and rng.group(2):
        fam = "press" if "cmh2o" in g or "cm h2o" in g else ("time_min" if "min" in g else None)
        return ("range", (rng.group(1), rng.group(2), fam))
    return ("other", None)
def comparator(gold, ids):
    kind, payload = parse_gold(gold); Q = ctx_quantities(ids)
    T = gnorm(" ".join(seed_render(n) + " " + " ".join(props_by.get(n, [])) for n in ids))
    if kind == "dim":
        a, b, c = payload; mm = {n for n, f in Q if f == "len_mm"}
        if {a, b, c} <= mm: return True
        t = T.replace(" ", "")
        return any(re.search(r"(?<!\d)" + p[0] + "x" + p[1] + "x" + p[2] + r"(?!\d)", t) for p in itertools.permutations([a, b, c]))
    if kind == "num":
        n, fam = payload
        if fam is None: return any(x == n for x, _ in Q)
        eq = FAM_EQ.get(fam, {fam}); return any(x == n and f in eq for x, f in Q)
    if kind == "range":
        a, b, fam = payload; t = T.replace(" ", "")
        if re.search(r"(?<!\d)" + a + r"\s*-\s*" + b, T) or (a + "-" + b) in t or (a + "to" + b) in t: return True
        if fam:
            eq = FAM_EQ.get(fam, {fam}); xs = {x for x, f in Q if f in eq}; return a in xs and b in xs
        return False
    if kind == "sdcard":
        t = T.replace(" ", ""); return ("sdcard" in t) and (">1year" in t or "1year" in t)
    return None
def word_overlap(gold, ids, thr=0.6):
    ng = gnorm(gold); ctx = gnorm(" ".join(units_of(ids)))
    w = set(re.findall(r"[a-z][a-z0-9\-]{2,}", ng)); cw = set(re.findall(r"[a-z][a-z0-9\-]{2,}", ctx))
    return bool(w) and len(w & cw) / len(w) >= thr
def router_present(gold, ids):
    r = comparator(gold, ids)
    if r is None: return bool(word_overlap(gold, ids))
    return bool(r)

# deterministic exact-value presence (H193/H194 robustness instrument; CPU-only, glyph/unit aware)
GLYPH = {'™':'', '®':'', '©':'', '–':'-', '—':'-', ' ':' ', ' ':' ',
         ' ':' ', 'ﬁ':'fi', 'ﬂ':'fl', '′':"'", '″':'"', '°':' ', '×':'x'}
def gnorm2(s):
    s = s or ""
    for k, v in GLYPH.items(): s = s.replace(k, v)
    s = unicodedata.normalize("NFKD", s); s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", s).casefold().strip()
_UNIT = r"(cmh2o|cm h2o|mm|cm|dba|db\(a\)|db|kg|g|oz|ml|l|w|hz|watts?|mins?|hours?|years?|m)"
def _numunits(t): return re.findall(r"(\d[\d,\.]*)\s*" + _UNIT + r"?", gnorm2(t))
def exact_present(gold, ctx):
    g = gnorm2(gold); c = gnorm2(ctx)
    if g and g in c: return True
    gd = re.sub(r"[ ,]", "", g); cd = re.sub(r"[ ,]", "", c)
    if any(ch.isdigit() for ch in gd) and len(gd) >= 4 and gd in cd: return True
    gnu = _numunits(gold)
    if gnu:
        for num, unit in gnu:
            nd = num.replace(",", "")
            if unit:
                if not (re.search(r"(?<!\d)" + re.escape(nd) + r"\s*" + re.escape(unit), c) or
                        re.search(re.escape(nd) + re.escape(unit), cd)): return False
            else:
                if not re.search(r"(?<!\d)" + re.escape(nd) + r"(?!\d)", cd): return False
        return True
    return False
def is_numeric_gold(g): return bool(re.search(r"\d", g))
def deterministic_present(gold, ids):
    ctx = " ".join(units_of(ids))
    if is_numeric_gold(gold): return exact_present(gold, ctx)
    if exact_present(gold, ctx): return True
    return word_overlap(gold, ids)
rprint("[green]scorers ready[/green] router (comparator+word-overlap)  |  deterministic (exact U word-overlap, CPU-only)")


scorers ready router (comparator+word-overlap)  |  deterministic (exact U word-overlap, CPU-only)

## Fingerprint assertion - MUST match the pinned H207 census before measuring

In [4]:
CANON_SPEC = dict(
    source="pipeline._retrieve_local entity_blocks (production query path)",
    per_seed=["## name (types)", "Also known as (SAME_AS*1..2, <=5)", "description",
              "Properties: json(prop_* keys, alias-merged)", "Relations: type -> name (<=15, non-SIMILAR_TO)"],
    proposition_channel="per-seed attached propositions (Proposition-[:ABOUT]->seed)",
    eval_k=K64, retrieve_top_k=RETRIEVE_TOPK, rel_limit=REL_LIMIT,
    seed_order="score desc, id asc (deterministic tie-break)",
    scorer="deterministic: exact_present(numeric/code) OR word_overlap>=0.6(prose); CPU-only, no NLI")
def render_fingerprint(spec, sample_ids):
    blob = json.dumps({k: spec[k] for k in sorted(spec)}, default=str) + "\x1e" + \
           "\x1e".join(seed_render(c) for c in sorted(sample_ids))
    return hashlib.sha256(blob.encode()).hexdigest()[:16]
def graph_content_hash(carrier_ids):
    return hashlib.sha256(("\x1e".join(seed_render(c) for c in sorted(carrier_ids))).encode()).hexdigest()[:16]
ALL_IDS = sorted(node)
RENDER_FP = render_fingerprint(CANON_SPEC, ALL_IDS)
CONTENT_HASH = graph_content_hash(ALL_IDS)
PINNED = dict(render_fp="96ab16d299fbbc71", content_hash="6fdc41bde495d1a3")
assert RENDER_FP == PINNED["render_fp"], f"render fingerprint drift: {RENDER_FP}"
assert CONTENT_HASH == PINNED["content_hash"], f"graph content drift: {CONTENT_HASH}"
rprint(f"[magenta]render_fingerprint[/magenta] {RENDER_FP}  [magenta]content_hash[/magenta] {CONTENT_HASH}  -> [green]MATCH pinned H207[/green]")

render_fingerprint 96ab16d299fbbc71  content_hash 6fdc41bde495d1a3  -> MATCH pinned H207

## H208 unit-conversion extension

Static conversion table to a per-dimension base unit (mass -> gram, length -> mm, pressure -> cmH2O,
flow -> L/min), 2% tolerance for rounding. The extension is **additive and cross-family only**: baseline
runs first; the conversion path fires only when a context quantity in the **same dimension but a different
unit** matches the gold within tolerance. Same-unit exact matching is untouched, so same-unit verdicts
cannot regress.

In [5]:
# ---- H208 dimension-aware unit conversion (mass/length/pressure/flow; static table; 2% tolerance) ----
# unit token -> (dimension, factor to base). base: mass=gram, len=mm, press=cmH2O, flow=L/min
XUNIT = {
 "g":("mass",1.0),"kg":("mass",1000.0),"oz":("mass",28.349523125),
 "lb":("mass",453.59237),"lbs":("mass",453.59237),"pound":("mass",453.59237),"pounds":("mass",453.59237),
 "mm":("len",1.0),"cm":("len",10.0),"inch":("len",25.4),"inches":("len",25.4),"in":("len",25.4),
 "cmh2o":("press",1.0),"hpa":("press",1.0197162),"mbar":("press",1.0197162),"kpa":("press",10.197162),
 "l/min":("flow",1.0),"lpm":("flow",1.0),"ml/min":("flow",0.001),
}
FAM_BASE = {  # comparator family -> (dimension, factor to base)
 "mass_g":("mass",1.0),"mass_kg":("mass",1000.0),"mass_oz":("mass",28.349523125),"mass_lb":("mass",453.59237),
 "len_mm":("len",1.0),"len_cm":("len",10.0),"len_in":("len",25.4),
 "press":("press",1.0),
}
TOL = 0.02
_XALT = r"(cmh2o|cm h2o|ml/min|l/min|mm|cm|kg|lbs|lb|pounds|pound|oz|inches|inch|hpa|mbar|kpa|lpm|in|g|m)"
def _close(a, b): return b != 0 and abs(a - b) / abs(b) <= TOL

def _x_text_quant(text):
    Q = set(); t = gnorm(text)
    for m in re.finditer(r"(\d+(?:\.\d+)?)\s*" + _XALT + r"\b", t):
        u = m.group(2).replace(" ", "")
        if u in XUNIT:
            dim, f = XUNIT[u]; Q.add((round(float(m.group(1)) * f, 4), dim, u))
    return Q
def x_ctx(ids):
    """context quantities converted to base units, tagged with dimension + source unit token."""
    Q = set()
    for nid in ids:
        spec = merged_spec(nid); unit_for = {}
        for k, v in spec.items():
            if k.endswith("_unit"):
                u = gnorm(str(v)).replace(" ", "")
                if u in XUNIT: unit_for[k[:-5]] = (XUNIT[u], u)
        for k, v in spec.items():
            db = None; fam = key_family(k)
            if fam and fam in FAM_BASE: db = (FAM_BASE[fam], "spec:" + fam)
            elif k in unit_for: db = unit_for[k]
            if db:
                (dim, f), utok = db
                for num in nums_in(v): Q.add((round(float(num) * f, 4), dim, utok))
        Q |= _x_text_quant(seed_render(nid) + " " + " ".join(props_by.get(nid, [])))
    return Q
def x_gold_scalar(gold):
    """scalar gold -> [(base_value, dim, unit_token), ...]; None if not a unit scalar / is a dim triple."""
    g = gnorm(gold)
    if re.search(r"\d+\s*x\s*\d+\s*x\s*\d+", g): return None
    out = []
    for m in re.finditer(r"(\d+(?:\.\d+)?)\s*" + _XALT + r"\b", g):
        u = m.group(2).replace(" ", "")
        if u in XUNIT:
            dim, f = XUNIT[u]; out.append((round(float(m.group(1)) * f, 4), dim, u))
    return out or None
def x_gold_dims(gold):
    """dimension triple -> (triple, unit_token); unitless triple returns ('' unit)."""
    g = gnorm(gold)
    mt = re.search(r"(\d+(?:\.\d+)?)\s*x\s*(\d+(?:\.\d+)?)\s*x\s*(\d+(?:\.\d+)?)\s*" + _XALT + r"?\b", g)
    if not mt: return None
    return [float(mt.group(i)) for i in (1, 2, 3)], (mt.group(4) or "").replace(" ", "")

# scale factor per token (for the genuine-conversion test in clause c)
SCALE = {u: XUNIT[u][1] for u in XUNIT}
SCALE.update({"spec:mass_g":1.0,"spec:mass_kg":1000.0,"spec:mass_oz":28.3495,"spec:mass_lb":453.59237,
              "spec:len_mm":1.0,"spec:len_cm":10.0,"spec:len_in":25.4,"spec:press":1.0})
def genuine_conv(gu, cu):
    # a real cross-unit conversion iff the two tokens carry different scale factors within the dimension
    return abs(SCALE.get(gu, 1.0) - SCALE.get(cu, 1.0)) > 1e-6

def comparator_x(gold, ids):
    """Extended comparator: baseline verbatim first; add a cross-family conversion path only."""
    base = comparator(gold, ids)
    if base is True: return True
    gs = x_gold_scalar(gold)
    if gs:
        Q = x_ctx(ids)
        for (gb, gdim, gu) in gs:
            for (cb, cdim, cu) in Q:
                if cdim == gdim and cu != gu and _close(gb, cb):
                    return True
    return base   # preserves None / False
def router_present_x(gold, ids):
    r = comparator_x(gold, ids)
    if r is None: return bool(word_overlap(gold, ids))
    return bool(r)

def values_consistent(v1, v2):
    """clause (a): True if two stored value strings are the same physical quantity under conversion."""
    d1 = x_gold_dims(v1); d2 = x_gold_dims(v2)
    if d1 and d2:
        (t1, u1), (t2, u2) = d1, d2
        f1 = XUNIT.get(u1, ("len", 25.4))[1] or 25.4   # unitless dim triple -> assume inches (imperial datasheet)
        f2 = XUNIT.get(u2, ("len", 25.4))[1] or 25.4
        b1 = sorted(x * f1 for x in t1); b2 = sorted(x * f2 for x in t2)
        return all(_close(a, b) for a, b in zip(b1, b2))
    s1 = x_gold_scalar(v1); s2 = x_gold_scalar(v2)
    if s1 and s2 and len(s1) == 1 and len(s2) == 1:
        (b1, dim1, _), (b2, dim2, _) = s1[0], s2[0]
        return dim1 == dim2 and _close(b1, b2)
    return None
rprint("[green]H208 conversion extension ready[/green] "
       f"dims=mass/len/press/flow  units={len(set(XUNIT))}  tol={TOL}")


H208 conversion extension ready dims=mass/len/press/flow  units=19  tol=0.02

## Clause (a) - reclassify the H204 unit-variant conflicts as consistent

In [6]:
conf = json.load(open(ROOT / "data/processed/probes-conflict-h204.json"))
clause_a = []
for p in conf["distinct"]:
    v1, v2 = p["values"]; cons = values_consistent(v1, v2)
    clause_a.append(dict(id=p["id"], attribute=p["attribute"], values=[v1, v2],
                         unit_variant=p["unit_variant"], genuine_conflict=p["genuine_value_conflict"],
                         reclassified_consistent=bool(cons)))
    rprint(f"  [bold]{p['id']}[/bold] {p['attribute']:12s} {v1!r} vs {v2!r}: "
           f"consistent={cons}  (unit_variant={p['unit_variant']} genuine_conflict={p['genuine_value_conflict']})")
variants = [c for c in clause_a if c["unit_variant"]]
genuine = [c for c in clause_a if c["genuine_conflict"]]
a_pass = all(c["reclassified_consistent"] for c in variants) and all(not c["reclassified_consistent"] for c in genuine)
rprint(f"[{'green' if a_pass else 'red'}]clause (a) {'PASS' if a_pass else 'FAIL'}[/]: "
       f"{sum(c['reclassified_consistent'] for c in variants)}/{len(variants)} unit-variants reconciled, "
       f"{sum(c['reclassified_consistent'] for c in genuine)}/{len(genuine)} genuine conflicts wrongly reconciled")

C001 weight       '3.5 lbs' vs '1.6kg': consistent=True  (unit_variant=True genuine_conflict=False)

C002 ramp time    '60mins' vs '10 minutes': consistent=None  (unit_variant=False genuine_conflict=True)

C003 dimensions   '8.66 × 7.6 × 4.4' vs '220 × 194 × 112 mm': consistent=True  (unit_variant=True 
genuine_conflict=False)

clause (a) PASS: 2/2 unit-variants reconciled, 0/1 genuine conflicts wrongly reconciled

## Clause (b) - frozen-bench replay (fixed ctx_ids, no retrieval): ZERO verdict changes

In [7]:
def replay_bench(pairs, name):
    flips = []
    for p in pairs:
        ids = [i for i in p["ctx_ids"] if i in node]
        b = router_present(p["gold"], ids); x = router_present_x(p["gold"], ids)
        if b != x:
            flips.append(dict(i=p.get("i"), gold=p["gold"], stratum=p.get("stratum"),
                              label=p.get("label"), base=b, ext=x,
                              false_credit=(p.get("label") == 0 and x and not b)))
    fc = sum(1 for f in flips if f["false_credit"])
    rprint(f"[bold]clause (b) {name}[/bold]: {len(pairs)} pairs, [{'green' if len(flips)==0 else 'yellow'}]"
           f"{len(flips)} verdict changes[/], [{'green' if fc==0 else 'red'}]{fc} false credits[/]")
    for f in flips: rprint("   FLIP", f)
    return flips, fc

h194 = json.load(open(ROOT / "data/processed/instrument-bench-h194.json"))["pairs"]
h196 = json.load(open(ROOT / "data/processed/instrument-prose-bench-h196.json"))["pairs"]
f194, fc194 = replay_bench(h194, "H194")
f196, fc196 = replay_bench(h196, "H196")
b_pass = (len(f194) == 0 and len(f196) == 0)
false_credits = fc194 + fc196
rprint(f"[{'green' if b_pass else 'yellow'}]clause (b): {len(f194)+len(f196)} total verdict changes[/], "
       f"[{'green' if false_credits==0 else 'red'}]{false_credits} false credits (REFUTE trigger)[/]")

clause (b) H194: 60 pairs, 0 verdict changes, 0 false credits

clause (b) H196: 48 pairs, 0 verdict changes, 0 false credits

clause (b): 0 total verdict changes, 0 false credits (REFUTE trigger)

## Clause (c) - wide-v2 retrieval replay: genuine cross-unit credits (reported either way)

In [8]:
qc = pickle.load(open(ROOT / "notebooks/.wide_probes_h188_qcache.pkl", "rb"))
def qkey(q): return hashlib.md5(q.encode()).hexdigest()
def seeds_from(p):
    res = vector_query(driver2, qc[qkey(p["question"])], VEC, top_k=RETRIEVE_TOPK)
    res = sorted(res, key=lambda x: (-x["score"], x["id"]))
    return [x["id"] for x in res if x["id"] in node][:K64]

W = json.load(open(ROOT / "data/processed/probes-wide-v2-h195.json"))["probes"]
covered = [p for p in W if qkey(p["question"]) in qc]
rprint(f"wide-v2 coverage: {len(covered)}/{len(W)} questions in qcache")
new_credits = []; parse_credits = []
for p in covered:
    ids = seeds_from(p)
    for g in p["gold_evidence"]:
        b = router_present(g, ids); x = router_present_x(g, ids)
        if x and not b:
            gs = x_gold_scalar(g); trig = None; genuine = False
            if gs:
                Q = x_ctx(ids)
                for (gb, gdim, gu) in gs:
                    for (cb, cdim, cu) in Q:
                        if cdim == gdim and cu != gu and _close(gb, cb):
                            trig = f"gold {g!r}({gu}->{gb}) ~ ctx {cb}({cu})"; genuine = genuine_conv(gu, cu); break
                    if trig: break
            rec = dict(id=p["id"], gold=g, rule=p.get("derivation_rule"), trigger=trig, genuine_conversion=genuine)
            (new_credits if genuine else parse_credits).append(rec)
rprint(f"[bold]clause (c)[/bold]: [green]{len(new_credits)} genuine cross-unit conversion credits[/]; "
       f"{len(parse_credits)} same-scale parse-coverage credits (excluded - not conversions)")
for c in new_credits: rprint("   CREDIT", c)
for c in parse_credits: rprint("   parse-only (excluded)", c)

wide-v2 coverage: 212/219 questions in qcache

clause (c): 0 genuine cross-unit conversion credits; 1 same-scale parse-coverage credits (excluded - not 
conversions)

parse-only (excluded)
{
    'id': 'V044',
    'gold': '4 cm H2O',
    'rule': 'spec_table_cell',
    'trigger': "gold '4 cm H2O'(cmh2o->4.0) ~ ctx 4.0(spec:press)",
    'genuine_conversion': False
}

### Clause (c) diagnostic - why the corpus yields no cross-unit credit

For every mass/length gold: is its carrier retrieved, does baseline already credit it, and what
unit does the graph store the same dimension in? The corpus stores weight natively in kg and
dimensions in mm; imperial-stated golds are already credited verbatim or their carriers are absent.

In [9]:
massdim = [p for p in covered if any(
    re.search(r"\d+(?:\.\d+)?\s*(kg|lbs?|oz|pounds?|inch|inches)\b", g.lower()) or
    re.search(r"\d+\s*x\s*\d+\s*x\s*\d+", g.lower()) for g in p["gold_evidence"])]
for p in massdim:
    ids = seeds_from(p); Q = x_ctx(ids)
    for g in p["gold_evidence"]:
        gs = x_gold_scalar(g); gd = x_gold_dims(g)
        if not (gs or gd): continue
        b = router_present(g, ids); gdim = (gs[0][1] if gs else "len")
        ctx_same = sorted({(cb, cu) for cb, cd, cu in Q if cd == gdim})[:6]
        rprint(f"  {p['id']} gold={g!r} base_present={b} dim={gdim} ctx[{gdim}]={ctx_same}")

V027 gold='1,33 kg' base_present=False dim=mass ctx=[(45.0, 'g'), (1106.0, 'g'), (1330.0, 'kg'), (1333.5616, 
'lbs'), (1360.0, 'kg'), (1360.7771, 'lbs')]

V049 gold='1.7 kg' base_present=True dim=mass ctx=[(45.0, 'g'), (1130.0, 'g'), (1133.9809, 'oz'), (1700.0, 'kg'),
(2000.0, 'kg'), (2000.3424, 'lbs')]

V053 gold='170 x 135 x 180 mm' base_present=True dim=len ctx=[(19.0, 'mm'), (19.0, 'spec:len_mm'), (22.0, 'mm'), 
(22.0, 'spec:len_mm'), (34.0, 'mm'), (38.0, 'mm')]

V065 gold='3.5 lbs' base_present=True dim=mass ctx=[(45.0, 'g'), (1106.0, 'g'), (1330.0, 'kg'), (1360.0, 'kg'), 
(1360.7771, 'lbs'), (1450.0, 'kg')]

V082 gold='1.36 kg' base_present=True dim=mass ctx=[(4.0, 'g'), (45.0, 'g'), (1106.0, 'g'), (1130.0, 'g'), 
(1133.9809, 'oz'), (1360.0, 'kg')]

V092 gold='40 oz' base_present=True dim=mass ctx=[(2.0, 'g'), (4.0, 'g'), (114.0, 'g'), (1130.0, 'g'), (1130.0, 
'spec:mass_g'), (1133.9809, 'oz')]

V123 gold='4.5 kg' base_present=True dim=mass ctx=[(1130.0, 'g'), (1133.9809, 'oz'), (3855.5351, 'pounds'), 
(4500.0, 'kg'), (19958.0643, 'lbs'), (29937.0964, 'lb')]

V157 gold='1.33 kg' base_present=True dim=mass ctx=[(1130.0, 'g'), (1133.9809, 'oz'), (1330.0, 'kg'), (1333.5616,
'lbs'), (1980.0, 'kg'), (1982.1987, 'lbs')]

V196 gold='1.60 kg' base_present=True dim=mass ctx=[(45.0, 'g'), (1330.0, 'kg'), (1450.0, 'kg'), (1600.0, 
'spec:mass_kg'), (1980.0, 'kg'), (2300.0, 'kg')]

## Machine-readable report

In [10]:
ts = datetime.datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
report = dict(
    round="R19", hypothesis="H208", utc=ts, graph="neo4j2",
    render_fingerprint=RENDER_FP, content_hash=CONTENT_HASH,
    compute="CPU-only, deterministic, no LLM",
    conversion_table=dict(mass_base="gram", len_base="mm", press_base="cmH2O", tolerance=TOL,
                          units=sorted(set(XUNIT))),
    clause_a=dict(description="H204 unit-variant conflicts reclassify as consistent",
                  detail=clause_a, passed=bool(a_pass)),
    clause_b=dict(description="ZERO verdict changes on frozen H194/H196 benches",
                  h194_pairs=len(h194), h196_pairs=len(h196),
                  h194_verdict_changes=len(f194), h196_verdict_changes=len(f196),
                  h194_false_credits=fc194, h196_false_credits=fc196,
                  h194_flip_detail=f194, h196_flip_detail=f196,
                  total_verdict_changes=len(f194)+len(f196), total_false_credits=false_credits,
                  passed=bool(b_pass)),
    clause_c=dict(description="additional gold credited via cross-unit conversion in wide-v2",
                  covered=len(covered), total=len(W),
                  genuine_cross_unit_credits=len(new_credits), detail=new_credits,
                  parse_coverage_credits_excluded=parse_credits),
    refuted=bool(false_credits > 0),
    verdict_recommendation=("REFUTED" if false_credits > 0 else
                            ("CONFIRMED" if (a_pass and b_pass) else "PARTIAL")),
    notes=("clause (a) and (b) are the mandatory bars; clause (c) is reported either way. "
           "clause (c) = 0 genuine credits confirms the registration's predicted-uncertain outcome: "
           "the corpus stores each attribute in one unit system (weight kg, dimensions mm), so no gold "
           "is stored ONLY in a converted unit that baseline misses. The V044 parse-only 'credit' "
           "(4 cmH2O ~ 4 press, same scale factor) is excluded - not a conversion."))
out = ROOT / f"reports/unit-conversion-h208-{ts}.json"
out.write_text(json.dumps(report, indent=2, default=str))
rprint(f"[green]wrote[/green] {out}")
rprint(f"[bold]VERDICT[/bold] {report['verdict_recommendation']}  "
       f"(a={a_pass} b={b_pass} false_credits={false_credits} clause_c_credits={len(new_credits)})")
driver2.close()

/tmp/ipykernel_2760363/563380503.py:1: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")


wrote ../reports/unit-conversion-h208-20260707T192616Z.json

VERDICT CONFIRMED  (a=True b=True false_credits=0 clause_c_credits=0)